In [2]:
#complex PDF parser for RAG text+images+OCR+Langchain documents
import os
import json
import fitz
import pdfplumber
import pandas as pd
from PIL import Image
from pathlib import Path
from langchain_core.documents import Document
print("All required libraries are imported successfully.")

All required libraries are imported successfully.


In [ ]:
from pathlib import Path
pdf_path = Path(r"D:\GEN AI BASICS\RAG\DATA-PARSING\Data\complex_rag_parsing_sample_with_sunny_image.pdf")
# Main output folder
output_dir = Path(r"D:\GEN AI BASICS\RAG\DATA-PARSING\Data\parsed_complex_pdf_output")
# Subfolders
image_dir = output_dir / "extracted_images"
page_image_dir = output_dir / "page_images"
# Create all folders
output_dir.mkdir(parents=True, exist_ok=True)
image_dir.mkdir(parents=True, exist_ok=True)
page_image_dir.mkdir(parents=True, exist_ok=True)

D:\GEN AI BASICS\RAG\DATA-PARSING\Data\parsed_complex_pdf_output
D:\GEN AI BASICS\RAG\DATA-PARSING\Data\parsed_complex_pdf_output\extracted_images
D:\GEN AI BASICS\RAG\DATA-PARSING\Data\parsed_complex_pdf_output\page_images


In [13]:
def run_ocr_on_image(image_path):
   #run OCR on the image and return the extracted text 
   try:
      import pytesseract
      img=Image.open(image_path)
      text = pytesseract.image_to_string(img)
      return text.strip()
   except Exception as e:
      return f"Error occurred while running OCR on {image_path}: {e}"

In [14]:
# 3. Extract Text + Images using PyMuPDF
def extract_text_and_images(pdf_path):
    doc = fitz.open(pdf_path)

    page_records = []
    image_records = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_number = page_index + 1

        # Extract normal selectable text
        text = page.get_text("text")

        # Extract page metadata-like info
        page_info = {
            "page_number": page_number,
            "text": text.strip(),
            "image_count": len(page.get_images(full=True)),
            "width": page.rect.width,
            "height": page.rect.height,
        }

        page_records.append(page_info)

        # Render full page as image for OCR
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        page_image_path = PAGE_IMAGE_DIR / f"page_{page_number:03d}.png"
        pix.save(str(page_image_path))

        # OCR full page image
        ocr_text = run_ocr_on_image(page_image_path)
        page_info["ocr_text"] = ocr_text
        page_info["page_image_path"] = str(page_image_path)

        # Extract embedded images
        images = page.get_images(full=True)

        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = doc.extract_image(xref)

            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_path = IMAGE_DIR / f"page_{page_number:03d}_image_{img_index + 1}.{image_ext}"

            with open(image_path, "wb") as f:
                f.write(image_bytes)

            image_ocr_text = run_ocr_on_image(image_path)

            image_records.append({
                "page_number": page_number,
                "image_index": img_index + 1,
                "image_path": str(image_path),
                "image_ext": image_ext,
                "image_ocr_text": image_ocr_text
            })

    doc.close()

    return page_records, image_records
